> **구분: 수업 핵심** — 강사 설명을 들은 뒤 수강생 실습 구간에서 진행합니다.

# 04. 제스처→안전 명령과 launch / 60분

**선행:** 07.Mediapipe_v2 gesture 결과와 ROS Twist

**이 노트북의 결과:** gesture allowlist·신선도·TTL을 적용하고 launch 종료와 watchdog STOP을 확인합니다.

[단원 README](README.md) · [용어집](GLOSSARY.md)


<!-- lesson-purpose:start -->
## 이 실습은 무엇을 위한 것인가요?

손 관측을 속도 명령으로 바꾸기 전에는 허용한 제스처인지, 점수가 유효한지, 관측이 오래되지 않았는지 검사해야 합니다. launch는 관련 node를 함께 시작하지만 이런 판단을 대신하지 않습니다. 입력 중단 후 정지를 실제로 확인합니다.

### 전체 과정에서 내 코드가 하는 일

```text
제스처 관측
  ↓
allowlist·confidence·age TODO
  ↓
controller
  ↓
속도
  ↓
watchdog 정지
  ↓
입력 재개 관찰
```

허용 조건과 경계값을 구현합니다. confidence가 없는 실제 관측을 임의의 모델 확신으로 꾸미지 않고, 고정 점수 체험은 별도 조건으로 표시합니다.

### 실행 전에 생각하고 결과로 설명하기

**질문:** 새 메시지가 안 오면 subscriber callback에서만 정지하면 되나요?

**설명:** 입력이 끊기면 callback 자체가 불리지 않을 수 있습니다. 별도 주기의 watchdog이 마지막 입력 시각을 검사해야 합니다. 입력 중단·zero 명령·재입력 시각을 기록해 의도한 동작인지 확인합니다.

이후의 TODO는 각 절에서 지정한 위치만 수정하고, 정답 해설의 이유를 자기 입력에 적용해 설명합니다. 예제 실행 성공·학생 함수 검증·실제 장치 관찰은 해당 절에서 요구하는 범위대로 나누어 기록합니다.
<!-- lesson-purpose:end -->


## A. 강사 설명 — 아직 코드를 실행하지 않습니다

**설명 자료:** [실습 전 이론·실행 안내 84~88쪽](../슬라이드/강의자료_3일차_v2.pdf#page=84) → [실습 시작 89쪽](../슬라이드/강의자료_3일차_v2.pdf#page=89) → ipynb 수행 → [실습 후 결과 해설 90~95쪽](../슬라이드/강의자료_3일차_v2.pdf#page=90).

강사는 아래 순서로 이론과 노트북 사용 방법을 설명합니다. 수강생은 입력과 예상 결과를 말로 예측합니다.

1. 이 실습은 무엇을 위한 것인가요?
2. 전체 과정에서 내 코드가 하는 일
3. 실행 전에 생각하고 결과로 설명하기
4. 앞 실습의 위치와 이번에 다시 필요한 내용
5. 시간 지도
6. 1. 관측과 명령을 분리한다
7. 2. TODO — Gesture allowlist
8. 작성 전에 확인할 입력·출력과 순서

노트북에서 `1~4절과 실행안내의 입력 중단 관찰`의 입력·처리·출력을 보여 주고, 실행 위치와 수정할 항목을 짚습니다. 입력 형식·API·실행 주의사항을 설명한 뒤, 아래 **B. 수강생 실습 시작**에서 실행을 시작하도록 안내합니다.

**완료 기준:** 허용 조건 검증·launch·입력 중단/zero/재입력 시각 기록

개별 기능의 출력은 TODO 전 완성 예제를 실행한 직후 관찰합니다. TODO를 조합한 결과·정답 해설·오류 비교는 실습 후 슬라이드에서 안내합니다. 실행 전에는 결과를 예측합니다.

**시간 배분:** 설명 20분 → 실습 35분 → 결과 해설·확인 5분.


## B. 수강생 실습 시작

강사의 설명과 실습 방법 시연이 끝나면 여기부터 진행합니다. 아래 이론 문단은 실습 중 참고할 설명입니다.

1. 이 노트북의 커널을 새로 시작하고 첫 준비·import 셀부터 실행합니다. 다른 단원의 `common` 모듈이 남아 있지 않도록 노트북별 새 커널을 사용합니다.
2. **TODO 전 따라 실행하기**의 완성 예제를 수정 없이 한 셀씩 실행하고 출력·관찰 설명을 확인합니다. 그런 다음 TODO에서 기능을 조합합니다. 터미널 명령은 표시된 호스트/컨테이너 터미널에 입력합니다.
3. 다음 v2 폴더의 파일을 수정·저장한 뒤 노트북의 학습자 검증 셀을 실행합니다: [exercises/gesture_contract.py](exercises/gesture_contract.py).
4. 실제 입력 실행·결과 저장을 마친 뒤 C에서 [결과 해설 90~95쪽](../슬라이드/강의자료_3일차_v2.pdf#page=90)으로 돌아가 출력·오류를 비교하고 완료 기록을 작성합니다. 참고 정답은 자신의 시도 후 강사 안내에 따라 비교합니다.

**실행 범위:** 1~4절과 실행안내의 입력 중단 관찰

**ROS 실행 전:** 오프라인 수기 서명 명부의 본인 번호를 `ROS_DOMAIN_ID`로 설정합니다. 아래 명령의 7은 예시입니다. 본인 A·B·C 터미널은 같은 번호를 사용하고, 공동 통신 실습만 강사가 지정한 동일 번호를 사용합니다. 자세한 절차는 [실행안내](실행안내.md#ros_domain_id--수기-서명-명부-번호)를 따릅니다.


<!-- notebook-restart:start -->
## 실습 시작 확인

현재 노트북의 Python 커널과 `/workspace` 자료 경로를 확인합니다. 새 커널이거나 커널을 재시작했다면 첫 설정·import 셀부터 순서대로 실행합니다. 이전 노트북의 변수·import는 자동으로 전달되지 않습니다. 저장한 노트북·소스와 `Data` 파일은 유지됩니다. 접속·재부팅 복구는 [Setup 안내](../01.Setup/README.md)를 참고합니다.

**커널 준비:** JupyterLab에서 **Python 3 (ipykernel)** 을 선택합니다. 새 커널에서는 이 노트북의 첫 Python 셀부터 실행합니다. 다른 단원의 모듈이 남아 있는 커널이라면 재시작한 뒤 준비 셀을 다시 실행합니다. 실행 경로는 `/workspace` 또는 현재 단원 폴더이고 Python은 `/opt/course/bin/python`입니다. 다른 단원과 실행 중인 커널을 공유하지 않습니다. 각 단원의 `common` 모듈 이름이 같아 이전 import가 남으면 다른 코드를 읽을 수 있습니다. `NameError`가 나면 필요한 초기화 셀부터 다시 실행합니다.

TODO는 지정된 `exercises/*.py`를 JupyterLab 편집기에서 수정·저장하고 **학습자 검증 셀** 을 다시 실행합니다. 이 노트북 안의 TODO는 해당 코드 셀을 직접 수정합니다. 미완성 상태의 `NotImplementedError`·검증 실패는 예상된 출발점입니다. 참고 정답 검증 통과는 학습자 파일의 통과와 별도입니다.

**ROS 준비 — 노트북의 Python 모의 실험과 실제 ROS 실행 구분:** 아래 Python 셀은 새 fixture와 loader를 직접 만들므로 앞 노트북의 커널 변수·TODO 결과가 필요 없습니다. 실제 node는 **같은 개인 컨테이너의 JupyterLab Terminal** 에서 실행합니다. Ollama는 필요 없습니다.

[01_ros_basics.ipynb의 3-1~3-3절](01_ros_basics.ipynb)에 workspace 빌드와 `source`의 설명이 있으며, 필요한 절차는 다음과 같습니다. 빌드는 `ros_ws/src`의 패키지를 `Data/build/08.ROS_v2/install`에 설치·연결합니다. 첫 사용 또는 install 파일이 없을 때, 패키지 설정·실행 이름이 바뀌었을 때 다음을 실행하고 성공을 확인합니다. 같은 자료 폴더·호환 이미지의 빌드 결과가 남아 있으면 재부팅만을 이유로 다시 빌드할 필요는 없습니다.

```bash
bash
cd /workspace
bash 08.ROS_v2/scripts/build_workspace.sh
```

**새 터미널 A·B·C마다** 아래를 적용합니다. 이 설정은 현재 터미널에만 적용되며 다른 터미널·Python 커널로 전파되지 않습니다. 7은 예시입니다. 로컬·원격 모두 오프라인 수기 서명 명부의 본인 번호로 바꿉니다.

```bash
bash
source /opt/ros/humble/setup.bash
source /workspace/Data/build/08.ROS_v2/install/setup.bash
export ROS_DOMAIN_ID=7 # 7을 본인 수기 서명 명부 번호로 변경
ros2 pkg executables nepes_ros_lab
```

`status_talker`, `image_monitor`, `safe_turtle_controller` 등 실행 이름이 보이면 준비된 것입니다. 기본 ROS는 첫 source, 수업 패키지는 두 번째 source가 찾게 합니다. `Package not found`는 build 성공·현재 install source를 확인합니다. 같은 실습의 모든 터미널은 같은 Domain 번호를 사용합니다. 로컬 GUI를 함께 관찰할 때는 두 수강생이 로컬 컨테이너의 화면을 함께 보고, 원격 개인 Domain에서 로컬 토픽이 보일 것으로 기대하지 않습니다. 이전 실습의 source/controller·카메라를 종료한 뒤 다음 실행을 시작합니다.

### 앞 실습의 위치와 이번에 다시 필요한 내용

[07.Mediapipe_v2/02_hand_gestures.ipynb의 1·2절](../07.Mediapipe_v2/02_hand_gestures.ipynb)에 손목0, PIP `(6,10,14,18)`, tip `(8,12,16,20)` 거리 비교와 `OPEN_PALM/FIST/POINT/UNKNOWN/NO_HAND` 상태가 있습니다. 필요한 연결 규칙은 **POINT→point_forward, OPEN_PALM→open_palm, 나머지→unknown→STOP** 입니다. POINT에 좌우 정보는 없습니다. [03_camera_opencv.ipynb의 1·2절](03_camera_opencv.ipynb)에 Image 촬영 시각과 age 검사가 있으며, 여기서는 관측 시각 `observed_ms`로 신선도를 계산합니다. Twist의 `linear.x`는 전후 속도, `angular.z`는 회전 속도입니다. 기본 실제 손 입력의 confidence는 null이라 이동을 거부하며, 아래 순수 Python TODO는 confidence 입력 없이 이름·시간 계약만 연습합니다. 이전 TODO 결과를 가져오지 않습니다.

<!-- notebook-restart:end -->

## 정답 활용 안내

> 교재 버전: v2 (실행 이미지 v1.0)

정답 확인 시점과 직접 시도 시간은 강사가 현장 진도·이해도에 맞춰 조절합니다. 검증 통과 전에도 강사 안내에 따라 정답을 참고할 수 있습니다. 정답과 해설을 비교한 뒤 자신의 TODO에 반영하고 다시 검증하세요. 정답 실행만으로 자신의 TODO가 완성되지는 않습니다.


> **구분: 수업 핵심**

수업 시작에 입력과 기대 결과를 예측하고, 종료 전 아래 두 완료 조건을 확인합니다.

## 이 노트북의 완료 확인과 단원 연결 질문

[작업대 상태 확인 사례](../공통사례.md) · [평가 워크북](../평가워크북.md)

- **이 노트북 완료:** 이 노트북에 표시된 핵심 TODO·검증·저장 결과를 확인합니다. 자습 노트북은 자습 범위만 확인합니다.
- **단원 전체 결과:** 입력 토픽·처리 결과·출력 명령을 확인하고 종료합니다.
- **단원 마지막 연결 질문:** 입력 노드를 중단하고 마지막 관측 시각, STOP, 복구 후 새 관측을 확인합니다. 운영자가 어느 로그부터 볼지 설명합니다.

각 단원의 마무리·비교 시간에 5분을 사용합니다. 별도 시간을 추가하지 않습니다. 마지막 5분은 저장·오류 복구에 우선 사용하고 심화 설명은 자습으로 넘깁니다. 결과와 짧은 이유를 `Data/outputs/course-review/<단원>/`의 개인 기록에 남깁니다. 자료가 부족하면 미측정으로 적으며 성공값을 만들어 넣지 않습니다.

[ROS 터미널 운영표](../08.ROS_v2/실행안내.md)를 사용하세요. 모의 입력은 이동·정지를, 실제 카메라는 인식·승인 또는 거부 이유를 확인합니다. 

08의 MediaPipe confidence=null 이동은 기본 controller에서 거부됩니다. 입력 중단 실험은 실행안내의 source·controller 분리 명령으로 진행합니다.

USB 카메라 실습은 로컬 환경에서 한 번만 실행하고 두 수강생이 로컬 화면을 함께 봅니다. 역할을 교대하고 각자 결과를 기록합니다. [공동 실습 안내](../공동카메라실습.md)를 따릅니다. 녹화 경로는 재생·자습·장애 대응용 선택 경로입니다.


> **구분: 수업 핵심** — 강사 설명을 들은 뒤 수강생 실습 구간에서 진행합니다.

## 시간 지도

행동 경계 10분 → allowlist/TTL 10분 → TODO 15분 → 실패 주입 10분 → build/launch 10분 → 종료 5분.


In [ ]:
# 구분: 수업 핵심 셀
from pathlib import Path
import importlib.util, json, shutil, sys

def find_course_root():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "08.ROS_v2").is_dir():
            return candidate
    raise RuntimeError("08.ROS_v2를 포함한 과정 루트를 찾지 못했습니다.")

ROOT = find_course_root()
LESSON = ROOT / "08.ROS_v2"
if str(LESSON) not in sys.path:
    sys.path.insert(0, str(LESSON))
OPTIONAL = {name: bool(importlib.util.find_spec(name)) for name in ('rclpy', 'cv2', 'cv_bridge')}
print({"course_root": str(ROOT), "optional_live_dependencies": OPTIONAL})

def load_file(module_name, relative_path):
    path = LESSON / relative_path
    spec = importlib.util.spec_from_file_location(module_name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


> **구분: 수업 핵심** — 강사 설명을 들은 뒤 수강생 실습 구간에서 진행합니다.

## 1. 관측과 명령을 분리한다

Recognizer의 `point_forward`는 관측 결과이고 속도가 아닙니다. Controller만 고정 표를 가집니다.

| gesture | linear.x | angular.z |
|---|---:|---:|
| point_forward | 0.8 | 0.0 |
| point_left/right | 0.0 | +1.2/-1.2 |
| open_palm | 0.0 | 0.0 |

Unknown/None, 오래된 시각, 미래 시각은 STOP입니다. TTL은 명령 만료 시각을 정하고 별도 watchdog은 명령이 만료되면 STOP을 발행합니다. Callback 예외로 node만 죽는 것은 마지막 속도를 지우지 못할 수 있습니다.


In [ ]:
# 구분: 수업 핵심 셀
CASES = [
    ("point_left", 1000, 1100),
    ("execute_anything", 1000, 1100),
    (None, 1000, 1100),
    ("point_forward", 1000, 1501),
    ("point_forward", 1200, 1100),
]
print("각 case가 승인인지 STOP인지 먼저 예상하세요:", CASES)


## TODO 전 따라 실행하기 — 코드 채우기 없음

> **구분: 수업 핵심**

**완성 예제 실행 → 출력 관찰 → TODO에서 조합** 순서입니다. 아래 예제를 위에서부터 그대로 실행하고 관찰 질문에 답하세요. 예제의 `warm_` 변수는 연습용입니다. 기존 실습 시간 안에서 먼저 진행하며, TODO 구현·검증 시간과 마지막 결과 정리 시간을 확보합니다.


### 따라 실행 1. in·dict 조회: 허용 목록과 속도

작은 고정 표에서 알려진 이름만 선택합니다. 아래 셀은 속도를 출력할 뿐 ROS로 명령을 발행하지 않습니다.


In [ ]:
# 완성 예제: 수정 없이 실행합니다.
warm_speeds = {'point_forward': (0.8, 0.0), 'point_left': (0.0, 1.2), 'point_right': (0.0, -1.2), 'open_palm': (0.0, 0.0)}
for warm_name in ('point_left', 'open_palm', 'unexpected', None):
    warm_known = isinstance(warm_name, str) and warm_name in warm_speeds
    print(warm_name, '알려짐:', warm_known, '속도:', warm_speeds[warm_name] if warm_known else (0.0, 0.0))


**실행 후 관찰:** open_palm은 알려진 이름이지만 속도는 0입니다. 승인 여부와 움직임을 구분하세요.


### 따라 실행 2. age·and·TTL: 조건을 합치기

알려진 입력이어도 관측 시각이 미래이거나 너무 오래되면 사용하지 않습니다.


In [ ]:
# 완성 예제: 수정 없이 실행합니다.
for warm_age in (-1, 0, 500, 501):
    warm_fresh = 0 <= warm_age <= 500
    print('age:', warm_age, 'known=True 승인:', True and warm_fresh, 'known=False 승인:', False and warm_fresh)
warm_now, warm_ttl = 1500, 500
print('issued_ms / expires_ms:', warm_now, warm_now + warm_ttl)
print('bool은 시간에서 제외:', isinstance(True, int), not isinstance(True, bool))


**실행 후 관찰:** TODO에서는 시간 타입 검사, 허용 목록, 신선도, 정지 이유와 반환 dict를 조합합니다. 이 학습 함수의 만료 시각은 now+TTL입니다.


### 이제 TODO에서 조합합니다

방금 실행한 기능을 아래 TODO의 입력·출력 계약에 맞게 연결하세요. 고정된 예제 값 대신 함수 인자 또는 지정된 실습 입력을 사용합니다. 필요한 import와 오류 처리도 직접 구성합니다. 아래 학습자 검증을 통과하고 자신의 결과를 저장한 뒤 실습 후 PDF 해설로 돌아갑니다.


> **구분: 수업 핵심** — 강사 설명을 들은 뒤 수강생 실습 구간에서 진행합니다.

## 2. TODO — Gesture allowlist

[exercises/gesture_contract.py](exercises/gesture_contract.py)의 `to_twist`를 구현합니다. 알려진 문자열이고 `0 ≤ age ≤ valid_for_ms`일 때만 표의 고정 속도를 반환합니다. Unknown/future/stale은 `accepted=False`, 속도 0입니다. 유효 시간은 양수여야 합니다.

<!-- todo-review:requirements:start -->
### 작성 전에 확인할 입력·출력과 순서

**입력 → 출력:** starter의 `to_twist`는 제스처 이름과 `observed_ms,now_ms,valid_for_ms`를 받아 `command,linear_x,angular_z,issued_ms,expires_ms,accepted,reason`의 dict를 반환합니다. 정확한 함수 이름·인자는 starter를 유지합니다.
**작성 흐름 힌트:** 밀리초 정수·양의 유효기간 검사 → 알려진 이름과 0≤age≤valid_for_ms를 각각 판단 → 허용된 경우만 속도표 선택 → issued_ms=now, expires_ms=now+valid_for_ms로 새 dict.
**속도표:** point_forward=(0.8,0), point_left=(0,1.2), point_right=(0,-1.2), open_palm=(0,0). 미지/누락은 command='stop', accepted=False, reason='unknown_or_missing'; 알려진 이름의 오래된/미래 입력은 reason='stale_or_future'. 허용은 reason='accepted'입니다.
**설명할 지점:** “open_palm은 유효하게 승인된 정지이므로 accepted=True입니다. 명령 승인 여부와 이동 여부는 다른 값입니다.”
**시간 계약·통과 기준:** 세 시간 인자는 bool을 제외한 정수이며 observed_ms·now_ms≥0, valid_for_ms>0이어야 합니다. 타입 위반은 `TypeError`, 범위 위반은 `ValueError`입니다. 알려지지 않은 값은 문자열 외 타입도 포함하며 오류 대신 STOP dict를 반환합니다. accepted=True이면 command는 입력 이름, 그 외는 소문자 `'stop'`입니다. 미지 이름이면서 오래되었어도 reason 우선순위는 `'unknown_or_missing'`입니다. 이 학습용 함수의 expires_ms는 승인 여부와 무관하게 `now_ms+valid_for_ms`입니다. 실제 controller는 관측 시각 기반 TTL을 추가 적용하므로 두 역할을 구분합니다. 아래 학습자 검증이 네 고정 속도, 시간 오류, age=0/500/501, future·unknown 정지를 모두 통과해야 합니다.
<!-- todo-review:requirements:end -->


In [ ]:
# 구분: 수업 핵심 셀
student = load_file("student_ros_gesture", "exercises/gesture_contract.py")
expected_speeds = {
    "point_forward": (0.8, 0.0),
    "point_left": (0.0, 1.2),
    "point_right": (0.0, -1.2),
    "open_palm": (0.0, 0.0),
}
for gesture, speed in expected_speeds.items():
    command = student.to_twist(gesture, observed_ms=1000, now_ms=1500)
    assert command["accepted"] and command["command"] == gesture
    assert (command["linear_x"], command["angular_z"]) == speed
    assert command["issued_ms"] == 1500 and command["expires_ms"] == 2000
for gesture, observed, now in CASES[1:]:
    command = student.to_twist(gesture, observed_ms=observed, now_ms=now)
    assert not command["accepted"]
    assert (command["linear_x"], command["angular_z"]) == (0.0, 0.0)
for bad_time_args in (
    {"observed_ms": True, "now_ms": 0, "valid_for_ms": 500},
    {"observed_ms": -1, "now_ms": 0, "valid_for_ms": 500},
    {"observed_ms": 0, "now_ms": True, "valid_for_ms": 500},
    {"observed_ms": 0, "now_ms": -1, "valid_for_ms": 500},
    {"observed_ms": 0, "now_ms": 0, "valid_for_ms": 0},
    {"observed_ms": 0, "now_ms": 0, "valid_for_ms": -1},
    {"observed_ms": 0, "now_ms": 0, "valid_for_ms": True},
    {"observed_ms": 0, "now_ms": 0, "valid_for_ms": 1.5},
):
    try:
        student.to_twist("open_palm", **bad_time_args)
    except (TypeError, ValueError):
        pass
    else:
        raise AssertionError(f"잘못된 시간 설정을 거부해야 합니다: {bad_time_args!r}")
at_start = student.to_twist("point_forward", observed_ms=1000, now_ms=1000, valid_for_ms=500)
at_end = student.to_twist("point_forward", observed_ms=1000, now_ms=1500, valid_for_ms=500)
after_end = student.to_twist("point_forward", observed_ms=1000, now_ms=1501, valid_for_ms=500)
assert at_start["accepted"] and at_end["accepted"] and not after_end["accepted"]
print("PASS: 전체 allowlist·고정 속도, 시간 자료형, 0/500ms 경계, stale/future/unknown STOP")


 > **구분: 수업 핵심**

## 참고 정답

강사가 안내하는 시점에 아래 참고 정답과 해설을 확인합니다. [solutions/gesture_contract.py](solutions/gesture_contract.py)에 입력 검사·계산·반환이 직접 구현되어 있으며, 아래 접힌 Markdown에도 같은 코드가 있습니다. 정답 검증과 학습자 TODO 검증은 별도입니다.


In [ ]:
# 구분: 수업 핵심 셀
reference = load_file("reference_ros_gesture", "solutions/gesture_contract.py")
assert reference.to_twist("point_forward", observed_ms=0, now_ms=500)["linear_x"] == 0.8
assert not reference.to_twist("point_forward", observed_ms=0, now_ms=501)["accepted"]
print("PASS: 참고 정답 검증")


### 정답 해설 — 입력부터 결과까지

**정답 구현 위치:** [solutions/gesture_contract.py](solutions/gesture_contract.py)의 `to_twist`입니다. 시간 값 검사→알려진 이름인지 확인→age 검사→허용 시 속도 선택 순서입니다. point_forward는 (0.8,0), point_left/right는 각속도 ±1.2, open_palm은 (0,0)입니다.

현재 단순 함수는 `0 <= age <= valid_for_ms`로 정확히 500ms까지 허용하고 501ms에는 zero로 닫습니다. 4일차 손 제어기의 엄격한 `< TTL` 경계와 구분합니다. 알 수 없는 이름과 오래된 입력은 모두 정지하지만 reason이 다릅니다.

실제 08 MediaPipe 입력은 confidence=null이므로 별도 controller가 이동을 거부합니다. 이 함수 검증의 허용 속도를 실제 카메라 이동 성공으로 해석하지 않습니다.

**확인 질문:** source만 멈추고 controller를 남겨두어야 watchdog 동작을 관찰할 수 있는 이유는 무엇인가요?

**적용·재검증:** 정답을 읽고 지정된 `exercises/` 파일의 TODO에 반영해 저장합니다. 노트북의 학습자 검증 셀을 다시 실행하고 경계 입력도 확인합니다. 참고 정답의 검증 통과는 학생 파일의 통과를 대신하지 않습니다. 이후 기존 실행·저장 셀에서 자신의 결과 경로를 확인합니다. 정답 알고리즘과 보조 함수는 표시된 solutions 파일에서 직접 읽습니다. 실제 카메라·ROS 실행은 제공 runtime을 사용하므로 학생 TODO 변경이 자동 반영된다고 가정하지 않습니다.

<!-- todo-review:answer:start -->
<details>
<summary>실행 없이 읽는 참고 정답과 단계별 주석</summary>

[실제 구현 파일](solutions/gesture_contract.py)에서 가져온 코드입니다. 이 Markdown 블록은 실행되지 않습니다. 정답 알고리즘은 아래 파일 본문에 직접 구현되어 있습니다. 학생 파일에서는 기존 함수 이름과 인자를 유지하세요. 아래에 포함된 보조 함수·상수는 알고리즘을 읽기 위한 문맥입니다.

```python
"""참고 정답: 이름과 유효 시간을 확인해 제한된 속도 또는 정지를 반환합니다.
알 수 없는 입력과 오래된 입력은 이유를 구분해 정지합니다.

입력 검사부터 결과 반환까지 이 파일에서 직접 확인합니다.
실제 카메라·ROS 실행용 공통 구현과 분리한 학습용 정답입니다.
"""

from __future__ import annotations
from typing import Any

_GESTURE_TO_SPEED = {
    "point_forward": (0.8, 0.0),
    "point_left": (0.0, 1.2),
    "point_right": (0.0, -1.2),
    "open_palm": (0.0, 0.0),
}


def to_twist(
    gesture: str | None,
    *,
    observed_ms: int,
    now_ms: int,
    valid_for_ms: int = 500,
) -> dict[str, Any]:
    """제스처를 제한된 속도 명령으로 바꾸며 모든 실패는 STOP으로 닫는다."""
    if any(isinstance(v, bool) or not isinstance(v, int) for v in (observed_ms, now_ms, valid_for_ms)):
        raise TypeError("시간과 valid_for_ms는 bool이 아닌 밀리초 정수여야 합니다")
    if observed_ms < 0 or now_ms < 0 or valid_for_ms <= 0:
        raise ValueError("시간은 0 이상이고 valid_for_ms는 양수여야 합니다")
    # 이름 허용 여부와 관측 신선도를 각각 계산합니다.
    # 이름이 알려져 있어도 미래·오래된 관측이면 승인할 수 없습니다.
    age = now_ms - observed_ms
    known = isinstance(gesture, str) and gesture in _GESTURE_TO_SPEED
    fresh = 0 <= age <= valid_for_ms
    # 유효한 정지(open_palm)도 승인됩니다. 승인 여부와 이동 여부는 다릅니다.
    accepted = known and fresh
    linear, angular = _GESTURE_TO_SPEED[gesture] if accepted else (0.0, 0.0)
    # 정지 이유를 구분하고 속도·발행 시각·만료 시각을 함께 반환합니다.
    # 이 학습 함수의 만료는 now+유효기간이며 실제 제어기의 관측 시각 기반 TTL과 구분합니다.
    reason = "accepted" if accepted else ("stale_or_future" if known else "unknown_or_missing")
    return {
        "command": gesture if accepted else "stop",
        "linear_x": linear,
        "angular_z": angular,
        "issued_ms": now_ms,
        "expires_ms": now_ms + valid_for_ms,
        "accepted": accepted,
        "reason": reason,
    }
```

</details>
<!-- todo-review:answer:end -->


> **구분: 수업 핵심** — 강사 설명을 들은 뒤 수강생 실습 구간에서 진행합니다.

## 3. Build와 launch

<!-- launch-core:start -->
### ros2 launch는 무엇을 해결하나요?

제스처로 거북이를 움직이려면 입력을 만드는 `gesture_source`, 입력을 검사해 속도를 만드는 `safe_turtle_controller`, 화면에서 움직이는 `turtlesim_node`가 함께 필요합니다. `ros2 run`은 실행 파일 하나를 시작할 때 쓰고, `ros2 launch`는 **어떤 실행 파일을 어떤 설정으로 함께 시작할지 적어 둔 구성** 을 읽습니다. 이 수업 예제에서는 각 `Node(...)`가 별도 프로세스를 시작합니다.

![launch 소스 수정, 빌드와 환경 적용, 노드 시작 및 제스처에서 거북이까지의 데이터 흐름](assets/ros2_launch_overview.svg)

launch 파일은 거북이의 이동 알고리즘이 아닙니다. 이동 규칙은 제어기 코드에 있으며 launch는 그 프로그램을 시작하고 설정값을 전달합니다. launch가 오류 없이 시작해도 카메라 입력이나 메시지 연결까지 성공했다는 뜻은 아닙니다. 노드 목록·토픽·실제 메시지를 이어서 확인해야 합니다.

#### 명령을 네 부분으로 읽기

```text
ros2 launch    nepes_ros_lab    gesture_turtle.launch.py    with_turtlesim:=false
실행 도구      패키지 이름     실행 구성을 적은 파일       이 실행에서 바꿀 인자
```

파일을 찾는 출발점은 현재 노트북 폴더가 아니라 **현재 터미널에 적용된 ROS 패키지 설치 경로** 입니다. 교재의 원본은 `08.ROS_v2/ros_ws/src/nepes_ros_lab/launch/`에 있습니다. 빌드는 이 패키지를 `Data/build/08.ROS_v2/install/`에 설치하고, `source .../install/setup.bash`는 현재 터미널이 그 설치를 찾게 합니다. 새 파일을 추가하거나 설치 구성을 바꾸면 다시 빌드하고 setup을 적용합니다.

#### 인자와 노드 설정의 연결

![with_turtlesim은 IfCondition에, use_sim_time은 bool ROS 파라미터에 연결되는 과정](assets/ros2_launch_arguments.svg)

기존 `gesture_turtle.launch.py`를 열어 아래 요소를 실제 코드에서 찾으세요.

| 코드 요소 | 이 예제에서 하는 일 | 직접 수정할 때 확인할 점 |
|---|---|---|
| `generate_launch_description()` | 실행할 action을 담은 `LaunchDescription`을 반환 | 파일을 import하는 것만으로 노드를 직접 실행하지 않기 |
| `DeclareLaunchArgument("with_turtlesim", default_value="true")` | 외부에서 받을 설정 이름과 기본값 선언 | 이름 오타, 기본값, 설명 확인 |
| `LaunchConfiguration("with_turtlesim")` | 실행 시점의 설정값 참조 | 일반 Python 문자열이나 즉시 평가된 bool이 아님 |
| `condition=IfCondition(with_turtlesim)` | turtlesim을 시작할지 결정 | `bool(LaunchConfiguration(...))`로 바꾸지 않기 |
| `ParameterValue(..., value_type=bool)` | `use_sim_time`을 bool 타입으로 전달 | 숫자 파라미터는 int/float 등 실제 타입에 맞추기 |
| `Node(package=..., executable=...)` | 패키지의 실행 파일을 찾아 시작 | 실행 파일명과 Python 파일명을 혼동하지 않기 |
| `parameters=[{"command_ttl_seconds": 0.5, ...}]` | 제어기에서 읽을 ROS 파라미터 설정 | 노드가 선언하고 실제로 사용하는 이름이어야 함 |
| `output="screen"` | 프로세스 출력을 터미널에 표시 | 창을 띄우는 GUI 옵션이 아님 |

**launch 인자와 ROS 파라미터는 다릅니다.** `with_turtlesim`은 launch가 노드 실행 여부를 정할 때만 사용합니다. `use_sim_time`은 launch에서 받은 뒤 각 노드의 동명 ROS 파라미터로 명시적으로 전달합니다. 선언만 추가하고 사용처를 연결하지 않으면 동작은 바뀌지 않습니다.

`with_turtlesim:=false`는 화면 노드만 생략합니다. 제스처 발행과 속도 명령 발행은 계속됩니다. 또한 turtlesim을 사용한다고 `use_sim_time:=true`로 바꾸지 않습니다. 이 예제는 기본값 `false`로 실제 시간을 사용하며, `true`는 유효한 `/clock` 발행원이 준비된 별도 환경에서 사용합니다.

`command_ttl_seconds`는 현재 launch 코드에서 `0.5`로 고정한 **노드 파라미터** 이고, launch 인자로 선언되어 있지 않습니다. 따라서 명령 뒤에 `command_ttl_seconds:=0.3`을 덧붙이는 것만으로 제어기의 TTL이 바뀌지 않습니다. 외부에서 바꾸려면 launch 인자를 선언하고 `LaunchConfiguration`을 `ParameterValue(..., value_type=float)`로 감싸서 해당 `parameters` 값에 연결해야 합니다. 실제 제어기는 TTL 범위도 검사하고 시작할 때 값을 읽으므로 변경 후 재시작이 필요합니다. 이번 실습에서는 제어기의 안전 기본값을 유지합니다.

#### 실행 전에 설정 목록 확인하기

아래는 **노드를 시작하지 않고** 제공되는 launch 인자와 기본값을 보는 명령입니다. 처음 사용하거나 새 launch 파일을 추가했다면 빌드부터 합니다.

```bash
bash
bash /workspace/08.ROS_v2/scripts/build_workspace.sh
source /opt/ros/humble/setup.bash
source /workspace/Data/build/08.ROS_v2/install/setup.bash
export ROS_DOMAIN_ID=7 # 7을 본인 수기 서명 명부 번호로 변경
ros2 launch nepes_ros_lab gesture_turtle.launch.py --show-args
```

`with_turtlesim`의 기본값 `true`, `use_sim_time`의 기본값 `false`를 확인합니다. `--show-args`로 구성을 읽을 수 있어도 실제 장치 연결까지 검증되는 것은 아닙니다.

**시작 순서에 대한 주의:** `LaunchDescription` 목록의 앞에 적었다고 해당 노드의 준비 완료까지 기다려 주지는 않습니다. 센서·서비스 준비가 필요한 프로그램은 재시도·준비 상태 확인을 설계해야 합니다. 전체 종료는 실행 터미널에서 Ctrl+C로 요청하지만 노드 하나의 예기치 않은 종료가 항상 나머지를 종료시키지는 않습니다. 잔여 노드와 종료 로그를 확인하세요.

핵심 개념을 읽었으면 아래의 기존 제스처 실행을 진행합니다. 직접 설정 파일을 작성하는 연습과 오류 해결표는 뒤의 **자습 — 나의 launch 파일 만들기** 에 있습니다.
<!-- launch-core:end -->

```bash
bash
bash /workspace/08.ROS_v2/scripts/build_workspace.sh
source /opt/ros/humble/setup.bash
source /workspace/Data/build/08.ROS_v2/install/setup.bash
export ROS_DOMAIN_ID=7 # 7을 본인 수기 서명 명부 번호로 변경
# 아래 두 launch 중 자신의 접속 방식에 맞는 하나만 실행
# 로컬 Jetson: turtlesim GUI 포함
ros2 launch nepes_ros_lab gesture_turtle.launch.py

# 원격 사용자: 같은 결정적 source/controller를 GUI 없이 실행
```

```bash
bash
source /opt/ros/humble/setup.bash
source /workspace/Data/build/08.ROS_v2/install/setup.bash
export ROS_DOMAIN_ID=7 # 7을 본인 수기 서명 명부 번호로 변경
ros2 launch nepes_ros_lab gesture_turtle.launch.py with_turtlesim:=false
```

다른 터미널에서 관측과 최종 속도 한 건을 화면과 Data에 함께 남깁니다.

```bash
bash
source /opt/ros/humble/setup.bash
source /workspace/Data/build/08.ROS_v2/install/setup.bash
export ROS_DOMAIN_ID=7 # 7을 본인 수기 서명 명부 번호로 변경
mkdir -p /workspace/Data/outputs/08.ROS_v2
ros2 topic echo /gesture/observation --once | tee /workspace/Data/outputs/08.ROS_v2/04-gesture-observation.txt
ros2 topic echo /turtle1/cmd_vel --once | tee /workspace/Data/outputs/08.ROS_v2/04-cmd-vel.txt
```

Controller는 수신 시각이 아니라 observation의 observed_ms부터 age와 TTL을 계산합니다.

07의 실제 Tasks runtime을 연결하는 headless 경로는 다음과 같습니다.

```bash
bash
source /opt/ros/humble/setup.bash
source /workspace/Data/build/08.ROS_v2/install/setup.bash
export ROS_DOMAIN_ID=7 # 7을 본인 수기 서명 명부 번호로 변경
ros2 launch nepes_ros_lab mediapipe_gesture.launch.py   with_turtlesim:=false with_recorded_source:=true
```

`POINT→point_forward`, `OPEN_PALM→open_palm`만 매핑합니다. 07 결과에는 좌우 방향이 없으므로 LEFT/RIGHT를 추측하지 않고 `FIST/UNKNOWN/NO_HAND`는 `unknown→STOP`으로 보냅니다. 현재 07 규칙 분류기는 별도 gesture confidence를 내지 않아 JSON에 `null`로 보존합니다. 기본 controller는 점수 없는 이동은 거부하고 점수 없는 `open_palm` 정지만 허용하며 camera stamp freshness와 watchdog도 검사합니다. 두 source를 동시에 실행하지 않습니다.

### 추가 체험 — USB 웹캠으로 turtlesim 전진·정지

**흐름:** USB 웹캠 → MediaPipe 손 특징점 → 제스처 규칙 → 체험용 `confidence=1.0` → ROS 제어기 → turtlesim.

기본 예제는 점수 없는 이동을 거부합니다. 이 별도 예제는 동작 연결을 체험하기 위해 정상 `POINT`·`OPEN_PALM`에 **고정값 1.0** 을 넣습니다. 모델의 확률이나 정확도 100%가 아닙니다. 영상 창에는 `DEMO confidence=1 (fixed)`, 상세 로그에는 `confidence_source=classroom_fixed`가 표시됩니다. 한 사람의 한 손만 화면에 넣습니다. 손바닥은 정지, 검지만 펴면 전진, 주먹·손 없음·판정 불가는 정지입니다. 좌우 회전 제스처는 이 예제에 없습니다.

로컬 사용자가 한 번 실행하고 두 수강생이 카메라 영상 창과 turtlesim을 함께 봅니다. 앞의 모의 제스처·기본 MediaPipe launch와 다른 카메라 프로그램을 종료하고 시작합니다. 같은 입력·제어 topic에 여러 source/controller를 동시에 실행하지 않습니다.

**처음 한 번 — 새 launch 파일을 반영해 빌드:** 로컬 컨테이너 Terminal에서 실행합니다.

```bash
bash
cd /workspace
bash 08.ROS_v2/scripts/build_workspace.sh
```

**터미널 A — 카메라 발행:** 장치 번호를 확인하고 실행합니다. 이후 입력 중단을 시험할 때 이 터미널만 Ctrl+C로 중단합니다.

```bash
bash
source /opt/ros/humble/setup.bash
source /workspace/Data/build/08.ROS_v2/install/setup.bash
export ROS_DOMAIN_ID=7 # 7을 본인 수기 서명 명부 번호로 변경
ros2 run v4l2_camera v4l2_camera_node --ros-args \
  -p video_device:=/dev/video0 -p image_size:="[640,480]" \
  -r image_raw:=/camera/image_raw
```

**터미널 B — 손 영상 창·인식·제어·turtlesim 실행:**

```bash
bash
source /opt/ros/humble/setup.bash
source /workspace/Data/build/08.ROS_v2/install/setup.bash
export ROS_DOMAIN_ID=7 # 7을 본인 수기 서명 명부 번호로 변경
ros2 launch nepes_ros_lab webcam_gesture_demo.launch.py
```

**터미널 C — 실제 출력 속도 확인:**

```bash
bash
source /opt/ros/humble/setup.bash
source /workspace/Data/build/08.ROS_v2/install/setup.bash
export ROS_DOMAIN_ID=7 # 7을 본인 수기 서명 명부 번호로 변경
ros2 topic echo /turtle1/cmd_vel
```

공동 실습의 세 터미널은 강사가 지정한 동일 명부 번호를 사용합니다. 이 공동 카메라 체험은 원격 개인 컨테이너에서 중복 실행하지 않습니다.

**관찰 순서:** 손바닥으로 정지 확인 → 검지만 펴서 전진 확인 → 주먹/손 숨기기로 정지 확인 → 전진 중 A만 중단하여 0.5초 TTL 뒤 정지 확인 → A를 다시 실행하고 새 손 입력으로 복구 확인. 제어기 timer 주기와 통신 지연만큼 관찰 시점이 더 늦을 수 있습니다. 거북이가 벽에 닿으면 전진 속도가 발행되어도 위치가 변하지 않으므로 속도 topic과 함께 판단합니다.

영상 창에서 `q`/Esc로 인식 node를 종료하면 입력이 끊겨 watchdog이 정지합니다. 전체 종료는 B에서 Ctrl+C, A에서 Ctrl+C 순으로 합니다. 상세 결과는 `Data/logs/08.ROS_v2/webcam-gesture-demo.jsonl`에 누적되며 `source_label`, `mapped_gesture`, `confidence`, `confidence_source`를 확인합니다. 실험 결과는 실제 관찰한 이동·정지와 지연을 기록합니다.

유효한 촬영 timestamp가 있는 영상이 오래되었거나 미래 시각이면 confidence가 1이어도 이동을 거부합니다. 이 예제의 고정 confidence를 실제 인식 신뢰도 평가에 사용하지 않습니다. 기본 `mediapipe_gesture.launch.py`는 기존처럼 `confidence=None`을 유지합니다.

**관측 시각의 예외:** `mediapipe_gesture.py`는 Image의 `header.stamp`가 0이면 처리 시점의 현재 시각으로 `observed_ms`를 채웁니다. 이 경우 TTL은 대체된 시각부터 계산하므로 실제 촬영 시각의 신선도는 확인하지 못합니다. 기본 `confidence=None` 경로의 이동 거부는 유지되지만, 고정 `confidence=1.0` 체험에서는 촬영 시각이 없는 입력도 최신 관측처럼 처리될 수 있습니다. 고정 confidence는 인식 정확도나 영상의 신선도를 보장하지 않습니다. 실제 수업 카메라에서 0 stamp가 발생하는지는 별도 관찰이 필요하며, 정상 stamp가 있는 영상 전체의 문제라는 뜻은 아닙니다.


> **구분: 수업 핵심** — 강사 설명을 들은 뒤 수강생 실습 구간에서 진행합니다.

## 4. 종료 계약

<!-- procedure:lesson-29-68c4e6:start -->
![입력 중단과 전체 종료 구분하기](../교재그림/절차/lesson-29.svg)

입력을 끊는 실험 중에는 제어기를 유지합니다. 제어기까지 먼저 종료하면 watchdog 관찰이 달라집니다.
<!-- procedure:lesson-29-68c4e6:end -->


입력 source 중단 → TTL 뒤 STOP 확인 → launch `Ctrl+C` → 마지막 zero Twist → node list 확인 순서입니다. Controller는 Python이 SIGINT/SIGTERM 요청을 먼저 받고 ROS context가 유효한 동안 zero Twist를 발행한 뒤 짧게 전송 기회를 주고 node/context를 정리합니다. 기본 `rclpy` signal handler가 context를 먼저 닫으면 마지막 STOP을 보낼 수 없으므로 이 순서를 코드와 검증에서 고정합니다. 이 속도는 turtlesim 교육값이며 실제 robot의 단위·가속도·충돌 방지·emergency stop을 대신하지 않습니다.

### 입력만 중단하는 watchdog 실험 — 필요한 명령을 여기서 다시 제공

[실행안내.md의 「입력만 중단하여 watchdog 확인하기」](실행안내.md)에 같은 절차가 있습니다. **통합 gesture/MediaPipe launch를 먼저 끝내고** 같은 로컬 컨테이너에서 A~D 터미널 네 개를 엽니다. 모든 터미널에 다음 준비를 각각 실행합니다.

```bash
bash
source /opt/ros/humble/setup.bash
source /workspace/Data/build/08.ROS_v2/install/setup.bash
export ROS_DOMAIN_ID=7 # 공동 실습: 강사가 지정한 동일 명부 번호
```

| 터미널 | 이어서 실행할 명령 | 역할 |
|---|---|---|
| A | `ros2 run turtlesim turtlesim_node` | 로컬 화면 |
| B | `ros2 run nepes_ros_lab safe_turtle_controller --ros-args -p command_ttl_seconds:=0.5` | 중단하지 않고 유지할 제어기 |
| C | `ros2 topic echo /turtle1/cmd_vel` | 실제 전진/zero 명령 관찰 |
| D | `ros2 run nepes_ros_lab gesture_source` | 이번에 중단할 입력만 실행 |

B·C 준비 후 D를 시작합니다. point_forward의 nonzero 속도를 보자마자 **D에서만 Ctrl+C** 를 누릅니다. B·C는 유지하고 마지막 승인 관측의 0.5초 TTL 뒤 zero Twist가 오는지 봅니다. D는 여섯 상태·마지막 open_palm을 내고 스스로 입력을 멈추므로 너무 늦게 중단하면 이미 정지해 있을 수 있습니다. 마지막 관측 시각·D 중단 시각·zero 관찰 시각을 함께 적고 zero의 반복 발행을 통과 조건으로 삼지 않습니다.

D를 다시 실행해 새로운 관측과 허용된 이동이 복구되는지 확인합니다. 전체 종료는 D→B→A→C 순서이며, 마지막에 `ros2 node list`와 카메라 해제를 확인합니다. 결과는 `Data/outputs/course-review/08.ROS_v2/`에 기록합니다. launch 전체를 종료한 것만으로 watchdog 실험 통과라고 쓰지 않습니다.




> **구분: 수업 후 복습·자습** — 상세 원리와 확장 내용입니다.

<!-- launch-practice:start -->
## 자습 — 나의 launch 파일 만들기 / 약 30–40분

**목표:** 노드 코드를 고치지 않고 노드 이름·토픽 이름·수신 노드의 실행 여부를 바꾸고, 실제 ROS 그래프에서 결과를 확인합니다. 카메라와 GUI를 사용하지 않으므로 로컬·원격 모두 개인 컨테이너에서 가능합니다. 이전 launch는 해당 터미널에서 Ctrl+C로 종료한 뒤 시작하세요.

![송신과 수신 노드 모두 factory status를 같은 상대 토픽으로 remap하여 team01 status로 통신하는 그림](assets/ros2_launch_remapping.svg)

### 1. 무엇을 바꾸는지 먼저 예측하기

| 설정 | 의미 | 이 실습의 값과 결과 |
|---|---|---|
| `package` / `executable` | 설치된 패키지 / 그 안의 실행 파일 | `nepes_ros_lab` / `status_talker`, `status_listener` |
| `name` | 실행 중 ROS 그래프에 나타날 노드 이름 | `sender`, `receiver` |
| `namespace` | 상대 ROS 이름 앞에 붙일 경로 | `team01` → `/team01/sender` |
| `remappings` | 코드에서 사용하는 통신 이름을 다른 이름으로 연결 | `/factory/status` → `status` → `/team01/status` |
| `condition` | 해당 action 실행 여부 | `with_listener:=false`면 수신 노드만 생략 |
| `ROS_DOMAIN_ID` | 발견·통신 영역을 정하는 프로세스 환경 | 양쪽 터미널에 본인 수기 서명 명부 번호 사용 |

namespace는 다른 학생과 완전히 분리하는 장치가 아닙니다. 개인 Domain 설정도 지켜야 합니다. `status_talker`와 `status_listener`의 원래 코드는 모두 **절대 이름** `/factory/status`를 사용하므로 namespace만 설정하면 토픽은 그대로입니다. 양쪽을 **상대 이름** `status`로 remap해야 `/team01/status`가 됩니다. 반대로 목적지를 `/shared/status`처럼 `/`로 시작하면 namespace가 앞에 붙지 않습니다.

### TODO 전 따라 실행하기 — launch 인자와 조건

아래 완성 셀은 ROS 환경을 적용한 별도 Python에서 실행합니다. 노드를 시작하지 않고 **LaunchConfiguration 값 읽기 → remap 쌍 구성 → IfCondition 평가**를 출력합니다. 코드 수정 없이 실행하세요.


In [ ]:
# 완성 예제: 수정 없이 실행합니다.
import subprocess
warm_launch_code = """
from launch import LaunchContext
from launch.substitutions import LaunchConfiguration
from launch.conditions import IfCondition
context = LaunchContext()
context.launch_configurations['namespace'] = 'demo'
context.launch_configurations['status_topic'] = 'state'
ns = LaunchConfiguration('namespace')
topic = LaunchConfiguration('status_topic')
print('namespace:', ns.perform(context))
print('remap:', [('/factory/status', topic.perform(context))])
for value in ('true', 'false'):
    context.launch_configurations['with_listener'] = value
    enabled = LaunchConfiguration('with_listener')
    print(value, '→ 실행 조건:', IfCondition(enabled).evaluate(context))
"""
warm_launch_run = subprocess.run(
    ['bash', '-c', 'source /opt/ros/humble/setup.bash && /usr/bin/python3 -'],
    input=warm_launch_code, text=True, capture_output=True, timeout=30)
print(warm_launch_run.stdout)
if warm_launch_run.stderr:
    print(warm_launch_run.stderr)
warm_launch_run.check_returncode()


**실행 후 관찰:** namespace는 demo, remap 목적지는 state이고 실행 조건은 True/False로 바뀝니다. `LaunchConfiguration`은 실행 환경에서 값을 읽는 객체입니다. TODO에서는 이 객체를 두 Node의 namespace·remappings에 연결하고 수신 Node에만 IfCondition을 전달합니다.

### 2. TODO — launch 파일 직접 작성하기

JupyterLab 파일 탐색기에서 `08.ROS_v2/ros_ws/src/nepes_ros_lab/launch/`에 **`my_status.launch.py`** 를 만들고 아래 코드를 붙여 넣습니다. 기존 파일을 덮어쓰지 않습니다. 노트북 코드 셀에서 실행하는 코드가 아닙니다.

현재 TODO 값은 문법상 실행 가능하지만 요구한 구성이 아직 연결되지 않은 상태입니다. 실행 전에 TODO 1–3을 고치세요.

```python
from launch import LaunchDescription
from launch.actions import DeclareLaunchArgument
from launch.conditions import IfCondition
from launch.substitutions import LaunchConfiguration
from launch_ros.actions import Node


def generate_launch_description():
    ns = LaunchConfiguration("namespace")
    topic = LaunchConfiguration("status_topic")
    enabled = LaunchConfiguration("with_listener")

    # TODO 1: ns를 두 노드가 공유하는 namespace로 연결하세요.
    node_namespace = ""
    # TODO 2: 원래 /factory/status를 topic으로 바꾸는 규칙 한 개를 넣으세요.
    topic_remaps = []
    # TODO 3: enabled에 따라 수신 노드 실행을 결정하세요.
    listener_condition = None

    return LaunchDescription([
        DeclareLaunchArgument("namespace", default_value="team01",
                              description="노드와 상대 토픽 이름의 접두 경로"),
        DeclareLaunchArgument("status_topic", default_value="status",
                              description="상태 메시지를 주고받을 상대 토픽 이름"),
        DeclareLaunchArgument("with_listener", default_value="true",
                              description="수신 노드도 시작할지 여부"),
        Node(package="nepes_ros_lab", executable="status_talker",
             name="sender", namespace=node_namespace,
             remappings=topic_remaps, output="screen"),
        Node(package="nepes_ros_lab", executable="status_listener",
             name="receiver", namespace=node_namespace,
             remappings=topic_remaps, condition=listener_condition,
             output="screen"),
    ])
```

**완성 조건:** 두 노드는 `team01`에 속하며 `/team01/status`로 통신해야 합니다. 외부 인자를 바꾸면 소스의 고정 문자열을 다시 고치지 않아도 결과가 바뀌어야 합니다. 수신 노드를 끄면 송신 노드는 계속 실행되어야 합니다.

### 3. 빌드하고 실행하기

수업 패키지의 `setup.py`에는 이미 `glob("launch/*.launch.py")`를 설치하는 항목이 있습니다. 지정한 폴더와 파일 확장자를 지키면 새 파일도 빌드 시 포함됩니다. Python 실행 파일을 새로 만든 경우에는 `console_scripts` 등록도 필요하지만, 이번에는 등록된 실행 파일을 재사용하므로 launch 파일만 추가합니다.

**터미널 A — 저장한 파일을 설치하고 기본 구성 시작:**

```bash
bash
bash /workspace/08.ROS_v2/scripts/build_workspace.sh
source /opt/ros/humble/setup.bash
source /workspace/Data/build/08.ROS_v2/install/setup.bash
export ROS_DOMAIN_ID=7 # 7을 본인 수기 서명 명부 번호로 변경
ros2 launch nepes_ros_lab my_status.launch.py --show-args
ros2 launch nepes_ros_lab my_status.launch.py
```

`publish sequence=...`와 `receive ...`가 반복됩니다. 출력 순서는 매번 다를 수 있습니다. 기본 송신 주기는 1초이고 카메라 영상이나 거북이 움직임은 없습니다.

**터미널 B — 실제 연결 확인:**

```bash
bash
source /opt/ros/humble/setup.bash
source /workspace/Data/build/08.ROS_v2/install/setup.bash
export ROS_DOMAIN_ID=7 # 7을 본인 수기 서명 명부 번호로 변경
ros2 node list
ros2 topic info /team01/status
ros2 topic echo /team01/status --once
```

`/team01/sender`, `/team01/receiver`, 메시지 타입 `std_msgs/msg/String`, publisher 1개와 subscriber 1개를 확인합니다. 이 수치는 다른 관찰 명령이 실행되지 않는 상태 기준입니다. `echo`도 구독자이므로 켜 놓으면 subscriber 수가 늘어납니다. 메시지의 `sequence`가 보이면 실제 데이터 전달까지 확인한 것입니다.

### 4. 인자만 바꿔서 다시 실행하기

A를 Ctrl+C로 종료한 뒤 **같은 A의 Bash** 에서 아래 한 줄을 실행합니다. 새 터미널을 열었다면 앞의 source 두 줄과 Domain 설정을 먼저 적용합니다.

```bash
ros2 launch nepes_ros_lab my_status.launch.py namespace:=team02 status_topic:=state with_listener:=false
```

예측한 결과와 실제 결과를 비교합니다.

| 확인할 항목 | 기대 결과 | 이유 |
|---|---|---|
| 노드 | `/team02/sender`만 있음 | 수신 노드의 `IfCondition`이 false |
| 토픽 | `/team02/state` | namespace와 상대 remap 목적지를 조합 |
| 송신 로그 | 계속 `publish sequence=...` | 조건은 수신 노드에만 적용 |
| 수신 로그 | 없음 | receiver 프로세스를 시작하지 않음 |
| subscriber 수 | 관찰 전 0개 | 수신 노드가 없고 echo도 아직 없음 |

B에서 확인하는 명령의 토픽도 `/team02/state`로 바꾸세요. `with_listener:=true`로 재시작하면 수신이 돌아옵니다. **namespace 변경과 Domain 변경을 혼동하지 마세요.** 두 터미널의 Domain은 계속 본인 수기 서명 명부 번호으로 유지합니다.

### 5. 정답과 자세한 해설

TODO 세 줄의 정답은 다음과 같습니다. 나머지 코드는 그대로 둡니다.

```python
node_namespace = ns
topic_remaps = [("/factory/status", topic)]
listener_condition = IfCondition(enabled)
```

1. `ns`는 `LaunchConfiguration("namespace")`입니다. 두 `Node`에 같은 참조를 전달하면 기본값 `team01`뿐 아니라 CLI에서 바꾼 `team02`도 두 노드에 적용됩니다. `"ns"`를 문자열로 넣으면 변수값이 아닌 글자 ns가 namespace가 됩니다.
2. remap 한 항목은 `(원래 이름, 목적지)` 쌍입니다. 두 노드의 원래 토픽이 `/factory/status`이므로 첫 값은 소스 코드와 정확히 맞춰야 합니다. 두 번째 값 `topic`은 실행 인자를 참조합니다. 송신 쪽만 바꾸면 수신자는 여전히 원래 통로를 듣기 때문에 메시지가 도착하지 않습니다. 메시지 타입과 내용은 remap으로 바뀌지 않습니다.
3. `IfCondition(enabled)`는 실행할 때 true/false를 평가합니다. `if enabled:` 또는 `bool(enabled)`는 설정 문자열의 참·거짓을 올바르게 평가하는 방법이 아닙니다. 이 조건을 listener에만 넣었으므로 송신은 계속됩니다.
4. `name="sender"`는 그래프에 표시될 이름을 바꿉니다. `executable="status_talker"`는 실행할 프로그램을 고릅니다. 실행 파일 이름까지 sender로 바꾸면 등록된 프로그램을 찾지 못합니다.

**제출할 관찰:** 기본 실행과 변경 실행 각각의 노드 목록·최종 토픽·송수신 여부를 적고, “수신 노드를 끄고도 publisher가 남는 이유”와 “절대 토픽은 namespace만으로 바뀌지 않는 이유”를 설명하세요. 마지막에는 A에서 Ctrl+C로 종료하고 잔여 노드가 없는지 확인합니다.

### 6. 오류별 조치와 더 확장할 때 볼 곳

| 증상 | 확인할 곳 | 조치 |
|---|---|---|
| `ros2: command not found` | 현재 컨테이너와 Bash 환경 | 수업 컨테이너의 Jupyter Terminal에서 ROS setup 적용 |
| `Package 'nepes_ros_lab' not found` | 빌드 결과와 현재 터미널의 install setup | 빌드의 첫 오류를 해결하고 overlay setup 다시 source |
| launch 파일을 찾을 수 없음 | 원본 위치·확장자·설치 경로 | `launch/my_status.launch.py`로 저장하고 재빌드. `ros2 pkg prefix --share nepes_ros_lab`로 찾는 패키지 위치 확인 |
| executable을 찾지 못함 | `ros2 pkg executables nepes_ros_lab` | `name`과 `executable`을 구분하고 등록된 실행 파일 사용 |
| 인자를 바꿨는데 동작이 같음 | `--show-args`, `LaunchConfiguration`, Node 사용처 | 선언한 인자가 실제 namespace/remap/condition/parameters에 연결됐는지 확인 |
| 노드는 있지만 메시지가 없음 | `ros2 node info /team01/receiver`, `ros2 topic info /team01/status --verbose` | 양쪽 Domain·최종 토픽·메시지 타입·QoS 확인. 남아 있는 기존 launch도 점검 |
| 예상과 달리 `/factory/status`가 남음 | 두 Node의 remappings | 절대 이름은 namespace만으로 안 바뀜. 양쪽 remap 누락 확인 |
| 파라미터 타입 오류 또는 값이 안 쓰임 | 노드의 `declare_parameter`와 `get_parameter` | 이름·타입·지원 여부 확인. launch에서는 `ParameterValue`로 타입 명시. 실행 중 변경 반영은 노드 구현에 달림 |
| `use_sim_time:=true` 이후 시간 진행 이상 | `/clock` 발행 여부 | 이 수업의 기본값 false로 복원하고 재시작 |
| 예시에 없는 오류 | launch 터미널의 첫 traceback, Data/build/08.ROS_v2/log, 아래 공식 문서 | 명령 전체·환경·첫 오류·패키지 경로를 함께 기록. 마지막 요약 에러만 보고 추측하지 않기 |

나중에 센서 설정을 추가할 때는 “노드가 이미 제공하는 ROS 파라미터인가 → launch 밖에서 바꿀 필요가 있는가 → 타입은 무엇인가 → 실제로 읽는 사용처는 어디인가” 순서로 확인하세요. 큰 시스템은 `IncludeLaunchDescription`으로 기존 launch를 묶고, 여러 노드의 공통 namespace는 `GroupAction`과 `PushRosNamespace`로 정리할 수 있습니다. 파라미터가 많아지면 YAML로 분리하고 설치 경로를 통해 읽는 방법으로 확장합니다. 고정 시간 대기는 준비 완료를 보장하지 않으므로 의존 노드의 준비 확인이 필요합니다.

**참고 자료:** [ROS 2 Humble launch 학습 안내](https://docs.ros.org/en/humble/Tutorials/Intermediate/Launch/Launch-Main.html), [launch 파일 작성과 패키지 설치](https://docs.ros.org/en/humble/Tutorials/Intermediate/Launch/Creating-Launch-Files.html), [실행 시점의 값 치환](https://docs.ros.org/en/humble/Tutorials/Intermediate/Launch/Using-Substitutions.html). 이 교재의 실제 구현은 [gesture launch](ros_ws/src/nepes_ros_lab/launch/gesture_turtle.launch.py), [설치 설정](ros_ws/src/nepes_ros_lab/setup.py), [송신 노드](ros_ws/src/nepes_ros_lab/nepes_ros_lab/status_talker.py), [수신 노드](ros_ws/src/nepes_ros_lab/nepes_ros_lab/status_listener.py)에서 확인합니다.
<!-- launch-practice:end -->

## 자습 — 제스처 떨림과 hysteresis / 약 45분

<!-- procedure:hysteresis-6fa191:start -->
![이동 진입과 유지 조건을 나누기](../교재그림/절차/hysteresis.svg)

시작 임계값과 유지 임계값은 다릅니다. 오래된 관측·입력 중단은 filtering과 관계없이 정지 조건입니다.
<!-- procedure:hysteresis-6fa191:end -->


Gesture는 계속 들어오는 관측이므로 Topic으로 전달합니다. 현재 제어 mode를 한 번 읽는 일은 Service, 목표 이동과 취소는 Action 후보입니다. 어떤 통신 방식을 쓰더라도 제스처 score가 threshold 근처에서 오르내리면 명령이 매 frame 바뀔 수 있습니다.

Hysteresis는 이동을 시작하는 threshold와 멈추는 threshold를 다르게 둡니다. 이 fixture는 0.80 이상에서 FORWARD를 시작하고, 이미 이동 중이면 0.75 미만일 때 STOP합니다. 입력이 사라지거나 오래되면 filtering 결과와 관계없이 즉시 STOP합니다.

진행 순서는 score sequence 예측 → single threshold와 hysteresis 비교 → 입력 중단 확인 → Data 기록입니다. 이 계산은 시간 규칙을 이해하기 위한 작은 모형이며 실제 controller source를 대신하지 않습니다.


In [ ]:
# 구분: 수업 후 복습·자습 셀
HYSTERESIS_SAMPLES = [
    {"id": "below_start", "confidence": 0.78, "fresh": True},
    {"id": "cross_start", "confidence": 0.81, "fresh": True},
    {"id": "small_drop", "confidence": 0.79, "fresh": True},
    {"id": "above_stop", "confidence": 0.76, "fresh": True},
    {"id": "below_stop", "confidence": 0.74, "fresh": True},
    {"id": "input_lost", "confidence": None, "fresh": False},
]
START_THRESHOLD = 0.80
STOP_THRESHOLD = 0.75
for sample in HYSTERESIS_SAMPLES:
    print(sample)


## TODO 전 따라 실행하기 — 코드 채우기 없음

> **구분: 수업 후 복습·자습**

**완성 예제 실행 → 출력 관찰 → TODO에서 조합** 순서입니다. 아래 예제를 위에서부터 그대로 실행하고 관찰 질문에 답하세요. 예제의 `warm_` 변수는 연습용입니다. 기존 실습 시간 안에서 먼저 진행하며, TODO 구현·검증 시간과 마지막 결과 정리 시간을 확보합니다.


### 따라 실행 1. 이전 상태·두 임계값: 상태를 이어 계산하기

다른 짧은 입력으로 STOP과 FORWARD 전환을 관찰합니다. 각 줄은 직전 상태를 이어받습니다.


In [ ]:
# 완성 예제: 수정 없이 실행합니다.
warm_state = 'STOP'
for warm_confidence, warm_fresh in [(0.80, True), (0.75, True), (0.74, True), (0.90, False)]:
    warm_before = warm_state
    if not warm_fresh:
        warm_state = 'STOP'
    elif warm_state == 'STOP' and warm_confidence >= 0.80:
        warm_state = 'FORWARD'
    elif warm_state == 'FORWARD' and warm_confidence < 0.75:
        warm_state = 'STOP'
    print(warm_before, warm_confidence, warm_fresh, '→', warm_state)


**실행 후 관찰:** 0.80에서 시작, 0.75에서는 유지, 0.74에서 정지합니다. 오래된 입력은 높은 점수여도 정지합니다. TODO에서는 여섯 입력의 상태를 이어 적습니다.


### 이제 TODO에서 조합합니다

방금 실행한 기능을 아래 TODO의 입력·출력 계약에 맞게 연결하세요. 고정된 예제 값 대신 함수 인자 또는 지정된 실습 입력을 사용합니다. 필요한 import와 오류 처리도 직접 구성합니다. 아래 학습자 검증을 통과하고 자신의 결과를 저장한 뒤 실습 후 PDF 해설로 돌아갑니다.


> **구분: 수업 후 복습·자습** — 상세 원리와 확장 내용입니다.

### TODO — 여섯 상태 예측

다음 셀의 None을 FORWARD 또는 STOP으로 바꾸세요. 첫 상태는 STOP에서 시작합니다.

1. 입력이 없거나 fresh가 False이면 즉시 STOP합니다.
2. 현재 STOP이면 confidence가 0.80 이상일 때만 FORWARD를 시작합니다.
3. 현재 FORWARD이면 confidence가 0.75 미만일 때만 STOP합니다.

0.80과 0.75의 등호가 어느 쪽에 포함되는지도 확인하세요.

**입력과 편집 위치:** 시간순 입력은 confidence `[0.78,0.81,0.79,0.76,0.74,None]`, fresh는 앞 다섯 개 True·마지막 False입니다. 아래 `MY_HYSTERESIS_TRACE`의 길이·순서를 유지하고 None 여섯 개만 대문자 문자열 `'FORWARD'` 또는 `'STOP'`으로 바꿉니다. 각 답은 해당 입력을 처리한 **뒤의 상태** 입니다. 매번 STOP에서 새로 계산하지 않고 직전 답을 현재 상태로 이어 갑니다. 시작 0.80은 FORWARD 포함, 이동 중 0.75는 유지입니다. 수정 셀→검증 셀을 다시 실행해 전체 trace PASS와 마지막 STOP을 확인하고 `04-gesture-hysteresis.json`에 저장합니다.


In [ ]:
# 구분: 수업 후 복습·자습 셀
MY_HYSTERESIS_TRACE = [
    None,  # TODO: below_start
    None,  # TODO: cross_start
    None,  # TODO: small_drop
    None,  # TODO: above_stop
    None,  # TODO: below_stop
    None,  # TODO: input_lost
]
print("현재 답:", list(zip(
    [sample["id"] for sample in HYSTERESIS_SAMPLES],
    MY_HYSTERESIS_TRACE,
)))


In [ ]:
# 구분: 수업 후 복습·자습 셀
def hysteresis_trace(samples, *, start_threshold, stop_threshold):
    state = "STOP"
    trace = []
    for sample in samples:
        confidence = sample["confidence"]
        if not sample["fresh"] or confidence is None:
            state = "STOP"
        elif state == "STOP" and confidence >= start_threshold:
            state = "FORWARD"
        elif state == "FORWARD" and confidence < stop_threshold:
            state = "STOP"
        trace.append(state)
    return trace

EXPECTED_HYSTERESIS_TRACE = hysteresis_trace(
    HYSTERESIS_SAMPLES,
    start_threshold=START_THRESHOLD,
    stop_threshold=STOP_THRESHOLD,
)
NAIVE_TRACE = [
    "FORWARD"
    if sample["fresh"] and sample["confidence"] is not None
    and sample["confidence"] >= START_THRESHOLD
    else "STOP"
    for sample in HYSTERESIS_SAMPLES
]
hysteresis_study_passed = False
if any(value is None for value in MY_HYSTERESIS_TRACE):
    print("TODO가 남았습니다. None 여섯 곳을 채우고 다시 실행하세요.")
else:
    assert all(value in {"FORWARD", "STOP"} for value in MY_HYSTERESIS_TRACE)
    assert MY_HYSTERESIS_TRACE == EXPECTED_HYSTERESIS_TRACE
    hysteresis_study_passed = True
    print("PASS: hysteresis 경계와 입력 중단 STOP을 구분했습니다.")
assert EXPECTED_HYSTERESIS_TRACE[-1] == "STOP"
print("single threshold:", NAIVE_TRACE)
print("hysteresis:", EXPECTED_HYSTERESIS_TRACE)


> **구분: 수업 후 복습·자습** — 상세 원리와 확장 내용입니다.

### 예상 결과와 해설

예상 trace는 STOP, FORWARD, FORWARD, FORWARD, STOP, STOP입니다.

- 0.81에서 시작 threshold를 넘어 이동합니다.
- 이후 0.79와 0.76은 0.75 이상이므로 바로 흔들려 정지하지 않습니다.
- 0.74는 stop threshold보다 작으므로 정지합니다.
- 마지막 입력 중단은 과거의 FORWARD를 유지하지 않고 즉시 STOP합니다.

Hysteresis는 오검출을 해결하거나 안전을 보장하지 않습니다. Freshness와 watchdog을 마지막 경계로 계속 유지해야 합니다.


In [ ]:
# 구분: 수업 후 복습·자습 셀
hysteresis_output = ROOT / "Data/outputs/08.ROS_v2/04-gesture-hysteresis.json"
hysteresis_output.parent.mkdir(parents=True, exist_ok=True)
hysteresis_record = {
    "exercise": "gesture_hysteresis",
    "start_threshold": START_THRESHOLD,
    "stop_threshold": STOP_THRESHOLD,
    "samples": HYSTERESIS_SAMPLES,
    "single_threshold_trace": NAIVE_TRACE,
    "hysteresis_trace": EXPECTED_HYSTERESIS_TRACE,
    "learner_trace": MY_HYSTERESIS_TRACE,
    "completed": MY_HYSTERESIS_TRACE == EXPECTED_HYSTERESIS_TRACE,
}
hysteresis_output.write_text(
    json.dumps(hysteresis_record, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
assert hysteresis_output.is_file()
print("saved:", hysteresis_output)
print("completed:", hysteresis_record["completed"])


## C. 실습 후 결과 해설·완료 확인

**슬라이드 복귀:** [실습 후 결과 해설 90~95쪽](../슬라이드/강의자료_3일차_v2.pdf#page=90).

실행한 셀의 출력·저장 결과를 열어 두고 강사 해설과 비교합니다. 예상과 달랐던 점·미완료 항목을 기록한 뒤 다음 이론으로 이동합니다.

허용 조건 검증·launch·입력 중단/zero/재입력 시각 기록

아래 항목을 본인의 결과로 채우고 노트북을 저장합니다.

| 항목 | 내 기록 |
|---|---|
| 실행한 입력·조건 | 작성 |
| 수정한 코드 또는 답안 | 작성 |
| 결과 파일·검증 출력 | 작성 |
| 예상과 달랐던 결과·이유 | 작성 |
| 미완료 항목·오류·다음 조치 | 작성 |

강사와 완료 기준을 확인한 후 다음 단원의 설명으로 이동합니다.
